# ADS 505 Technical Notebook: Predicting U.S. Flight Departure Delays

**Team:** 4 (individual project, instructor-approved)  
**Author:** Michael Rosalia  
**GitHub repository:** https://github.com/mrosalia96/ads505-flight-delay-prediction

Predict pre-departure delay risk (≥ 15 minutes) for flights originating at SAN, LAX, PHX, DEN, and SEA in 2023, using BTS on-time performance and NOAA weather. Sections are added as the analysis progresses.

In [ ]:
# All package imports live in this first code cell.
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_STATE = 42
PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
BTS_DIR = RAW_DIR / "bts"
NOAA_DIR = RAW_DIR / "noaa"

ORIGIN_AIRPORTS = ("SAN", "LAX", "PHX", "DEN", "SEA")
ANALYSIS_YEAR = 2023

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from build_flights_sample import build_sample, default_sample_path  # noqa: E402

print("Project root:", PROJECT_ROOT)

## 1. Problem statement and data description

**Problem.** Airlines and airports often learn about departure delays as they unfold. This project predicts **pre-departure delay risk** (15+ minutes late) so operations leaders can prioritize staffing, buffers, and passenger communication before pushback.

**Data.** U.S. DOT BTS Reporting Carrier On-Time Performance for calendar year 2023, filtered to origin airports SAN, LAX, PHX, DEN, and SEA. NOAA weather will be joined in a later step. Acquisition details and APA citations: `docs/data_sources.md`.

**How data are obtained (reproducibility).** This is **not** a live filtered API pull. Peers download the same public monthly ZIP files from fixed TranStats URLs (`src/download_bts.py`), then run `src/build_flights_sample.py` to keep only our five origins. Raw ZIPs and the processed parquet stay on disk (gitignored); the repo stores the scripts and docs so anyone can rebuild the sample and re-run this notebook.

**Target.** Prefer BTS `DepDel15` (1 = departure delay ≥ 15 minutes). Rows with missing delay outcome or cancelled flights are handled explicitly below.

In [ ]:
# Prefer an existing processed sample; rebuild only if missing.
# To force a rebuild: build_sample(year=ANALYSIS_YEAR, origins=ORIGIN_AIRPORTS)
# Requires raw ZIPs from: python src/download_bts.py --year 2023
sample_path = default_sample_path(ANALYSIS_YEAR)
if sample_path is None:
    sample_path = build_sample(year=ANALYSIS_YEAR, origins=ORIGIN_AIRPORTS)
print("Sample file:", sample_path)

In [ ]:
# Load the processed sample and report basic shape / delay rate.
flights = pd.read_parquet(sample_path) if sample_path.suffix == ".parquet" else pd.read_csv(
    sample_path, parse_dates=["FlightDate"]
)

# Modelling set: exclude cancelled flights; require a defined 15-min delay flag.
mask_usable = (flights["Cancelled"] == 0) & flights["DepDel15"].notna()
flights_model = flights.loc[mask_usable].copy()
flights_model["DepDel15"] = flights_model["DepDel15"].astype(int)

print(f"Raw filtered rows: {len(flights):,}")
print(f"Usable (not cancelled, DepDel15 known): {len(flights_model):,}")
print(f"Date range: {flights_model['FlightDate'].min().date()} → {flights_model['FlightDate'].max().date()}")
print("Rows by origin:")
print(flights_model["Origin"].value_counts().reindex(list(ORIGIN_AIRPORTS)))
print(f"Overall DepDel15 rate: {flights_model['DepDel15'].mean():.3%}")
flights_model.head()

## 2. Exploratory data analysis (EDA)

Charts appear only where discussed. Focus: where delay risk concentrates by **airport**, **month**, **scheduled hour**, **day of week**, and **carrier** — patterns an operations lead would recognize without weather yet.

### 2.1 Delay rate by origin airport

In [ ]:
delay_by_origin = (
    flights_model.groupby("Origin", as_index=False)["DepDel15"]
    .mean()
    .rename(columns={"DepDel15": "delay_rate"})
)
delay_by_origin["Origin"] = pd.Categorical(
    delay_by_origin["Origin"], categories=list(ORIGIN_AIRPORTS), ordered=True
)
delay_by_origin = delay_by_origin.sort_values("Origin")

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=delay_by_origin, x="Origin", y="delay_rate", color="#4C72B0", ax=ax)
ax.set_ylabel("Share of flights delayed ≥ 15 min")
ax.set_xlabel("Origin airport")
ax.set_title("Departure delay rate by origin (2023)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
for bar, rate in zip(ax.patches, delay_by_origin["delay_rate"]):
    ax.annotate(f"{rate:.1%}", (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()
delay_by_origin

DEN has the highest 15+ minute departure-delay rate (~27%), while SEA is lowest (~17%). SAN sits near the middle (~20%). A single national average would hide that hub-level risk differs by roughly 10 percentage points across this sample.

### 2.2 Seasonality by month

In [ ]:
delay_by_month = (
    flights_model.groupby("Month", as_index=False)["DepDel15"]
    .mean()
    .rename(columns={"DepDel15": "delay_rate"})
)

fig, ax = plt.subplots(figsize=(8, 4))
sns.lineplot(data=delay_by_month, x="Month", y="delay_rate", marker="o", ax=ax)
ax.set_xticks(range(1, 13))
ax.set_ylabel("Share of flights delayed ≥ 15 min")
ax.set_xlabel("Month")
ax.set_title("Departure delay rate by month (2023)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.set_ylim(0, max(0.35, delay_by_month["delay_rate"].max() * 1.15))
plt.tight_layout()
plt.show()
delay_by_month

Delay risk peaks in **June–July** (~28–29%) and is lowest in **November** (~15%). Summer operational pressure shows up clearly in this five-airport sample; a model that ignores month will miss a large, predictable swing.

### 2.3 Time-of-day pattern

In [ ]:
flights_model = flights_model.copy()
flights_model["crs_dep_hour"] = (flights_model["CRSDepTime"] // 100).astype("Int64")

delay_by_hour = (
    flights_model.dropna(subset=["crs_dep_hour"])
    .groupby("crs_dep_hour", as_index=False)
    .agg(delay_rate=("DepDel15", "mean"), n_flights=("DepDel15", "size"))
)
# Keep hours with enough volume for a stable rate
delay_by_hour = delay_by_hour[delay_by_hour["n_flights"] >= 500]

fig, ax = plt.subplots(figsize=(9, 4))
sns.lineplot(data=delay_by_hour, x="crs_dep_hour", y="delay_rate", marker="o", ax=ax)
ax.set_ylabel("Share of flights delayed ≥ 15 min")
ax.set_xlabel("Scheduled departure hour (local)")
ax.set_title("Departure delay rate by scheduled hour (2023)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
plt.tight_layout()
plt.show()
delay_by_hour.sort_values("delay_rate", ascending=False).head(5)

Early banks are comparatively reliable (roughly **7–11%** delayed at 05:00–07:00 scheduled), while late afternoon / evening banks reach about **28–31%**. That asymmetry matters for staffing: the same airport needs different attention at 6 a.m. vs 6 p.m.

### 2.4 Day of week

In [ ]:
dow_labels = {
    1: "Mon",
    2: "Tue",
    3: "Wed",
    4: "Thu",
    5: "Fri",
    6: "Sat",
    7: "Sun",
}

delay_by_dow = (
    flights_model.groupby("DayOfWeek", as_index=False)["DepDel15"]
    .mean()
    .rename(columns={"DepDel15": "delay_rate"})
)
delay_by_dow["day"] = delay_by_dow["DayOfWeek"].map(dow_labels)

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=delay_by_dow, x="day", y="delay_rate", color="#4C72B0", ax=ax,
            order=[dow_labels[i] for i in range(1, 8)])
ax.set_ylabel("Share of flights delayed ≥ 15 min")
ax.set_xlabel("Day of week")
ax.set_title("Departure delay rate by day of week (2023)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
plt.tight_layout()
plt.show()
delay_by_dow

Thursday, Friday, and Sunday sit at the high end (~23%), while Tuesday is lowest (~19%). The day-of-week swing is real but smaller than the airport and time-of-day gaps, so it is a useful secondary feature rather than the main driver.

### 2.5 Reporting airline (high-volume carriers)

In [ ]:
carrier_stats = (
    flights_model.groupby("Reporting_Airline", as_index=False)
    .agg(delay_rate=("DepDel15", "mean"), n_flights=("DepDel15", "size"))
)
# Stable rates only: carriers with at least 10,000 departures in the sample
carrier_stats = carrier_stats[carrier_stats["n_flights"] >= 10_000].sort_values(
    "delay_rate", ascending=False
)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=carrier_stats, x="Reporting_Airline", y="delay_rate", color="#4C72B0", ax=ax)
ax.set_ylabel("Share of flights delayed ≥ 15 min")
ax.set_xlabel("Reporting airline")
ax.set_title("Departure delay rate by carrier (≥10k flights)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
plt.tight_layout()
plt.show()
carrier_stats

Among high-volume carriers, delay rates still differ a lot: Southwest (WN) is near ~29%, while SkyWest (OO) is near ~14%. Carrier identity is informative for triage, but it can also proxy route mix and airport exposure, so we will treat it as a feature and check that it does not simply recreate the origin effect later.

### 2.6 EDA implications for modeling

Origin, month, and scheduled hour are the strongest schedule-side signals; day of week and reporting airline add secondary structure. Next: preprocess these fields, join origin weather, then split **by time** so summer peaks are not leaked into later test months.

## 3. Data pre-processing

Goal: freeze a clean modeling table with **pre-departure** features only (no delay-cause fields), document missingness and outliers, and check pairwise correlation among numeric candidates. Weather columns will be added in a later step before the train/validation/test split.

### 3.1 Missing values and basic cleaning

In [ ]:
# Ensure hour feature exists if earlier cells were run out of order
if "crs_dep_hour" not in flights_model.columns:
    flights_model = flights_model.copy()
    flights_model["crs_dep_hour"] = (flights_model["CRSDepTime"] // 100).astype("Int64")

feature_candidates = [
    "Month",
    "DayOfWeek",
    "crs_dep_hour",
    "Origin",
    "Dest",
    "Reporting_Airline",
    "Distance",
    "CRSElapsedTime",
]
target_col = "DepDel15"

missing_report = (
    flights_model[feature_candidates + [target_col]]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .rename("pct_missing")
)
print("Missingness among candidate fields:")
print(missing_report.map(lambda x: f"{x:.3%}"))

# Drop rows missing any candidate feature or the target (rare after cancel filter)
before = len(flights_model)
flights_clean = flights_model.dropna(subset=feature_candidates + [target_col]).copy()
after = len(flights_clean)
print(f"\nRows before dropna: {before:,}")
print(f"Rows after dropna:  {after:,} (removed {before - after:,})")

Cancelled flights and rows without `DepDel15` were already removed. Remaining candidate-feature missingness is expected to be very low; any leftover incomplete rows are dropped so train/val/test shares a consistent schema.

### 3.2 Outliers on distance and elapsed time

In [ ]:
numeric_check = ["Distance", "CRSElapsedTime", "crs_dep_hour"]
summary = flights_clean[numeric_check].describe(percentiles=[0.01, 0.05, 0.95, 0.99]).T
print(summary)

# Flag extreme schedule length / distance for review (do not auto-delete yet)
q01 = flights_clean["Distance"].quantile(0.01)
q99 = flights_clean["Distance"].quantile(0.99)
extreme_dist = (flights_clean["Distance"] < q01) | (flights_clean["Distance"] > q99)
print(f"\nDistance outside 1st–99th percentile: {extreme_dist.sum():,} "
      f"({extreme_dist.mean():.2%}) — kept for now; tree models handle heavy tails well.")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.histplot(flights_clean["Distance"], bins=40, ax=axes[0], color="#4C72B0")
axes[0].set_title("Distance distribution")
axes[0].set_xlabel("Distance (miles)")
sns.histplot(flights_clean["CRSElapsedTime"].dropna(), bins=40, ax=axes[1], color="#4C72B0")
axes[1].set_title("Scheduled elapsed time")
axes[1].set_xlabel("CRS elapsed time (minutes)")
plt.tight_layout()
plt.show()

Distance and scheduled block time are right-skewed (short-haul heavy with a long-haul tail), which is expected for these hubs. We keep extreme distance rows rather than winsorizing: they are real markets, and tree-based models can split on them. Linear models can use a log-distance transform later if needed.

### 3.3 Correlated numeric features

In [ ]:
corr_cols = ["Month", "DayOfWeek", "crs_dep_hour", "Distance", "CRSElapsedTime", target_col]
corr = flights_clean[corr_cols].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
ax.set_title("Correlation among numeric candidates and target")
plt.tight_layout()
plt.show()

print(
    "Distance vs CRSElapsedTime correlation:",
    f"{corr.loc['Distance', 'CRSElapsedTime']:.3f}",
)

`Distance` and `CRSElapsedTime` are highly correlated (same underlying trip length). For the baseline feature set we keep **Distance** and drop scheduled elapsed time to reduce redundancy. Month, day-of-week, and hour are weakly related to the target individually — useful in combination with airport/carrier and weather.

### 3.4 Feature list freeze (pre-weather)

In [ ]:
MODEL_FEATURES = [
    "Month",
    "DayOfWeek",
    "crs_dep_hour",
    "Origin",
    "Dest",
    "Reporting_Airline",
    "Distance",
]
LEAKAGE_EXCLUDED = [
    "DepDelay",
    "DepDelayMinutes",
    "DepTime",
    "CarrierDelay",
    "WeatherDelay",
    # and other post-departure / delay-cause fields not retained in the sample
]

flights_model_ready = flights_clean[MODEL_FEATURES + [target_col, "FlightDate"]].copy()
flights_model_ready[target_col] = flights_model_ready[target_col].astype(int)

print("Frozen features:", MODEL_FEATURES)
print("Explicitly excluded as leakage / post-outcome:", LEAKAGE_EXCLUDED)
print(f"Modeling table shape: {flights_model_ready.shape}")
print(f"Positive class rate: {flights_model_ready[target_col].mean():.3%}")
flights_model_ready.head()

The pre-weather modeling table uses only information knowable before departure: calendar, schedule hour, airports, airline, and distance. Actual departure times and delay-cause codes stay out.

### 3.5 Join origin-hour NOAA ASOS weather

Hourly ASOS observations (temperature, dewpoint, wind, precip, visibility) are pulled for each origin airport via the Iowa Environmental Mesonet interface to the NOAA ASOS network (`src/download_weather.py`), then aggregated to local hour (`src/build_weather_hourly.py`). We join on origin + flight date + scheduled departure hour.

In [ ]:
weather_parquet = PROCESSED_DIR / f"weather_hourly_{ANALYSIS_YEAR}.parquet"
weather_csv = PROCESSED_DIR / f"weather_hourly_{ANALYSIS_YEAR}.csv"
if weather_parquet.exists():
    weather = pd.read_parquet(weather_parquet)
elif weather_csv.exists():
    weather = pd.read_csv(weather_csv, parse_dates=["weather_date"])
else:
    raise FileNotFoundError(
        f"Missing weather file. Run src/download_weather.py and "
        "src/build_weather_hourly.py first. Also: pip install pyarrow"
    )

weather["weather_date"] = pd.to_datetime(weather["weather_date"])

flights_wx = flights_model_ready.merge(
    weather,
    left_on=["Origin", "FlightDate", "crs_dep_hour"],
    right_on=["Origin", "weather_date", "weather_hour"],
    how="left",
)

WEATHER_FEATURES = ["tmpf", "dwpf", "sknt", "p01i", "vsby"]
match_rate = flights_wx["tmpf"].notna().mean()
print(f"Weather match rate: {match_rate:.2%}")

before = len(flights_wx)
flights_wx = flights_wx.dropna(subset=WEATHER_FEATURES).copy()
print(
    f"Rows with complete weather: {len(flights_wx):,} "
    f"(dropped {before - len(flights_wx):,})"
)

MODEL_FEATURES_WX = MODEL_FEATURES + WEATHER_FEATURES
print("Features with weather:", MODEL_FEATURES_WX)
flights_wx[WEATHER_FEATURES].describe().T

About 99.7% of usable flights match an origin-hour weather observation. The small unmatched remainder (nighttime gaps or missing METARs) is dropped so every training row has the same weather fields.

### 3.6 Weather vs delay (quick check)

In [ ]:
wx_eda = flights_wx.copy()
wx_eda["wind_bin"] = pd.qcut(wx_eda["sknt"], 3, labels=["low", "mid", "high"])
wx_eda["had_precip"] = wx_eda["p01i"].fillna(0) > 0
wx_eda["low_vsby"] = wx_eda["vsby"] < 3

wind_rates = wx_eda.groupby("wind_bin", observed=True)["DepDel15"].mean()
precip_rates = wx_eda.groupby("had_precip")["DepDel15"].mean()
vsby_rates = wx_eda.groupby("low_vsby")["DepDel15"].mean()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
wind_rates.plot(kind="bar", ax=axes[0], color="#4C72B0", rot=0)
axes[0].set_title("Delay rate by wind tertile")
axes[0].set_ylabel("Share delayed ≥ 15 min")
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))

precip_rates.plot(kind="bar", ax=axes[1], color="#4C72B0", rot=0)
axes[1].set_title("Any hourly precip")
axes[1].set_xlabel("Precipitation in hour")
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))

vsby_rates.plot(kind="bar", ax=axes[2], color="#4C72B0", rot=0)
axes[2].set_title("Visibility < 3 miles")
axes[2].set_xlabel("Low visibility")
axes[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))

plt.tight_layout()
plt.show()

print("Wind tertiles:\n", wind_rates.map(lambda x: f"{x:.1%}"))
print("Precip:\n", precip_rates.map(lambda x: f"{x:.1%}"))
print("Low visibility:\n", vsby_rates.map(lambda x: f"{x:.1%}"))

Worse weather lines up with higher delay risk: high-wind hours (~25%) exceed low-wind hours (~19%), hours with precip jump to ~32% vs ~21% without, and low visibility (~30%) exceeds clearer conditions (~21%). Weather belongs in the model, not just as a narrative aside.

## 4. Data splitting (train, validation, test)

Use a **chronological** split so the model is judged on future flights, not a random shuffle that leaks summer patterns into winter. Working cutoffs for 2023: train Jan–Aug, validation Sep–Oct, test Nov–Dec.

In [ ]:
train_end = pd.Timestamp("2023-08-31")
val_end = pd.Timestamp("2023-10-31")

train_mask = flights_wx["FlightDate"] <= train_end
val_mask = (flights_wx["FlightDate"] > train_end) & (flights_wx["FlightDate"] <= val_end)
test_mask = flights_wx["FlightDate"] > val_end

train_df = flights_wx.loc[train_mask].copy()
val_df = flights_wx.loc[val_mask].copy()
test_df = flights_wx.loc[test_mask].copy()


def split_summary(name, frame):
    return {
        "split": name,
        "rows": len(frame),
        "share": len(frame) / len(flights_wx),
        "delay_rate": frame["DepDel15"].mean(),
        "start": frame["FlightDate"].min().date(),
        "end": frame["FlightDate"].max().date(),
    }


split_table = pd.DataFrame(
    [
        split_summary("train", train_df),
        split_summary("validation", val_df),
        split_summary("test", test_df),
    ]
)
print(split_table.to_string(index=False))
assert train_df["FlightDate"].max() < val_df["FlightDate"].min()
assert val_df["FlightDate"].max() < test_df["FlightDate"].min()
print("\nChronological order checks passed.")

Train covers the busy summer peak; validation and test are later months with lower baseline delay rates. That is intentional and harder than a random split: the model must generalize forward in time. Hyperparameters will be chosen on validation only; the test set stays untouched until final selection.